In [1]:
import pandas as pd

In [ ]:
latest_ods = pd.read_csv("../../pipeline/pipeline_steps/input_files/2012-01-2025-03-overdoses.csv")

/tmp/ipykernel_565479/4034167975.py:1: DtypeWarning: Columns (11,28,36,46) have mixed types. Specify dtype option on import or set low_memory=False.
  latest_ods = pd.read_csv("../../pipeline/pipeline_steps/input_files/2012-01-2025-03-overdoses.csv")


In [15]:
one_with_missing_2023 = pd.read_csv("../../pipeline/pipeline_steps/input_files/2012-01-2025-03-overdoses-0106.csv")

/tmp/ipykernel_565479/1038753858.py:1: DtypeWarning: Columns (11,28,36,46) have mixed types. Specify dtype option on import or set low_memory=False.
  one_with_missing_2023 = pd.read_csv("../../pipeline/pipeline_steps/input_files/2012-01-2025-03-overdoses-0106.csv")


In [3]:
older_ods = pd.read_csv("../../pipeline/pipeline_steps/input_files/2012-01-2025-01-overdoses_old.csv")

/tmp/ipykernel_565479/854716496.py:1: DtypeWarning: Columns (11,28,38,46) have mixed types. Specify dtype option on import or set low_memory=False.
  older_ods = pd.read_csv("../../pipeline/pipeline_steps/input_files/2012-01-2025-01-overdoses_old.csv")


In [4]:
latest_ods['DeathDate']  = pd.to_datetime(latest_ods['DeathDate'])

In [5]:
latest_ods_2023 = latest_ods[(latest_ods['DeathDate'] > '2023-05-31') & (latest_ods['DeathDate'] < '2023-12-31')]

In [6]:
older_ods['DeathDate']  = pd.to_datetime(older_ods['DeathDate'])

In [7]:
older_ods_2023 = older_ods[(older_ods['DeathDate'] > '2023-05-31') & (older_ods['DeathDate'] < '2023-12-31')]

In [8]:
older_ods_2023['CaseNumber']

16731    2023-06398
16732    2023-06410
16733    2023-06433
16734    2023-06442
16735    2023-06448
            ...    
18532    2024-00071
18533    2024-00077
18546    2024-00175
18722    2024-01274
19241    2024-04655
Name: CaseNumber, Length: 1785, dtype: object

In [9]:
merged = older_ods_2023.merge(latest_ods_2023, on='CaseNumber', how='outer', indicator=True)

missing_in_latest_ods_2023 = merged[merged['_merge'] == 'left_only']
missing_in_older_ods_2023 = merged[merged['_merge'] == 'right_only']

In [10]:
missing_in_latest_ods_2023['source_file_x'].value_counts()

Series([], Name: count, dtype: int64)

In [20]:
def overdoses_per_year(df, deathdate_col="DeathDate"):
    # Parse DeathDate safely
    dt = pd.to_datetime(df[deathdate_col], errors="coerce")
    # Count rows per year
    return dt.dt.year.value_counts(dropna=True).sort_index()

# Replace df1 and df2 with your actual dataframe names
counts_1 = overdoses_per_year(latest_ods)
counts_2 = overdoses_per_year(older_ods)
count_3 = overdoses_per_year(one_with_missing_2023)

# Combine into one table
overdose_table = pd.concat([counts_1, counts_2, count_3], axis=1).fillna(0).astype(int)
overdose_table.columns = ["Latest File (0107)", "File before 2023 data missing", "Previous file"]

# Make it a nice table with Year as a column
overdose_table = overdose_table.reset_index().rename(columns={"index": "Year"})

overdose_table

,DeathDate,Latest File (0107),File before 2023 data missing,Previous file
0,2012.0,527,528,527
1,2013.0,620,625,620
2,2014.0,642,636,642
3,2015.0,826,821,826
4,2016.0,871,871,871
5,2017.0,978,974,978
6,2018.0,1191,1186,1191
7,2019.0,1404,1398,1404
8,2020.0,2400,2395,2400
9,2021.0,2921,2915,2921


In [ ]:
import pandas as pd

def overdoses_per_year_flag(df, flag_col, deathdate_col="DeathDate"):
    """
    Counts overdoses per year where df[flag_col] indicates presence of a drug.
    Works with 1/0, True/False, Y/N, Yes/No, etc.
    """
    # Parse DeathDate safely
    dt = pd.to_datetime(df[deathdate_col], errors="coerce")

    # Create a robust "is flagged" mask
    flagged = df[flag_col].astype(str).str.strip().str.lower().isin(
        ["1", "true", "t", "y", "yes"]
    )

    # Filter dates to flagged rows only
    dt_flagged = dt[flagged]

    # Count rows per year
    return dt_flagged.dt.year.value_counts(dropna=True).sort_index()


def make_flag_table(flag_col, deathdate_col="DeathDate"):
    counts_1 = overdoses_per_year_flag(latest_ods, flag_col, deathdate_col)
    counts_2 = overdoses_per_year_flag(older_ods, flag_col, deathdate_col)
    counts_3 = overdoses_per_year_flag(one_with_missing_2023, flag_col, deathdate_col)

    overdose_table = pd.concat([counts_1, counts_2, counts_3], axis=1).fillna(0).astype(int)
    overdose_table.columns = ["Latest File (0107)", "File before 2023 data missing", "Previous file"]

    overdose_table = overdose_table.reset_index().rename(columns={"index": "Year"})
    return overdose_table


# --- Create the two tables ---
fentanyl_table = make_flag_table("Fentanyl")
meth_table = make_flag_table("Methamphetamine")

fentanyl_table

(    DeathDate  Latest File (0107)  File before 2023 data missing  \
 0      2012.0                   5                              5   
 1      2013.0                   9                              9   
 2      2014.0                  12                             12   
 3      2015.0                  12                             12   
 4      2016.0                  58                             58   
 5      2017.0                 111                            111   
 6      2018.0                 216                            216   
 7      2019.0                 411                            411   
 8      2020.0                1078                           1077   
 9      2021.0                1615                           1617   
 10     2022.0                1823                           1824   
 11     2023.0                1793                           1788   
 12     2024.0                1417                            706   
 13     2025.0                 283

In [25]:
fentanyl_table

,DeathDate,Latest File (0107),File before 2023 data missing,Previous file
0,2012.0,5,5,5
1,2013.0,9,9,9
2,2014.0,12,12,12
3,2015.0,12,12,12
4,2016.0,58,58,58
5,2017.0,111,111,111
6,2018.0,216,216,216
7,2019.0,411,411,411
8,2020.0,1078,1077,1078
9,2021.0,1615,1617,1615


In [ ]:
latest_ods['Race'] = latest_ods['Race'].replace({"middleeasternornorthafrican": "MIDDLE EASTERN", "cardiovasculardiseasenull": np.nan, "ck": np.nan, "k": np.nan})

In [ ]:
latest_ods.to_csv("../../pipeline/pipeline_steps/input_files/2012-01-2025-03-overdoses-cleaned-1219.csv")

In [ ]:
old_latest.columns

In [ ]:
KEYS = ["CaseNumber"]

In [ ]:
removed = (
    old_latest
    .merge(
        latest_ods[KEYS],
        on=KEYS,
        how="left",
        indicator=True
    )
    .query('_merge == "left_only"')
    .drop(columns="_merge")
)

print(len(removed))

In [ ]:
removed['CaseNumber']

In [ ]:
removed

In [ ]:
cases_24_new[cases_24_new['CaseNum'] == '2024-00046']['InjuryDesc']

In [ ]:
cases_24[cases_24['CaseNum'] == '2024-00046']['InjuryDesc']

In [ ]:
cases_24['InjuryDesc']